# GLM-5.3-Flash NVFP4 on 2x B200 (TP2), stock vLLM nightly

| Metric | Value |
|---|---|
| Boot | **275 s** (weights cached) |
| Bulk throughput, c=128, thinking on | ~2,100-2,400 tok/s (**inferred**, chars/4 estimate) |
| MoE backend | Marlin (weight-only NVFP4), auto-selected -- no `--moe-backend` flag |
| Thinking | cannot be disabled on this checkpoint |

Status: **research preview**. c=1/8/16 and MTP-3 are pending. See
[`recipe.md`](recipe.md) for the full writeup, including the job 1-4
post-mortem that led to this recipe.

This notebook is self-sufficient: it replays the operator's own boot and
sanity-probe findings (no GPU needed to read them), then gives a Modal
launch path and a plain `vllm serve` + `curl` path for a bare-metal 2x B200
box.

In [ ]:
# --- Status cell ---
EXPERIMENT = "glm-5.3-flash-nvfp4-b200-tp2"
LIVE = False  # this notebook replays committed operator notes/receipts; no bulk run executes here

print(f"experiment : {EXPERIMENT}")
print(f"LIVE       : {LIVE}")
print("status     : measured boot (275s) + sanity probe pass + inferred bulk tok/s.")
print("             c=1/8/16 and MTP-3 are untested (pending).")

In [ ]:
# --- Pins ---
pins = {
    "image_digest": "vllm/vllm-openai@sha256:41d42cfabd3289f40fdca71b4a0fe290474880d20e9534a90c698eb78e3eecee",
    "image_note": "same stock nightly digest as the Qwen3.8-Flash-Next lane in this repo",
    "model": "LibertAIDAI/GLM-5.3-Flash-NVFP4",
    "model_revision": "11d73216cd636238e82e1d77fe1042ffab36e7fa",
    "topology": "TP2, 2x B200",
    "server_flags": [
        "--max-model-len 20480",
        "--max-num-seqs 128",
        "--gpu-memory-utilization 0.90",
        "--trust-remote-code",
        "# NOTE: no --moe-backend, no --quantization -- both auto-detected, see Pitfalls",
    ],
    "sampling": {"temperature": 1.0, "top_p": 0.95, "top_k": "not set"},
    "reasoning_parser": "none for raw <think> capture; --reasoning-parser deepseek_r1 "
                        "is community-reported (Libertai) for serving",
    "thinking": "cannot be disabled -- chat template emits <think> unconditionally; "
                "knob is reasoning_effort low/high/max, default max",
    "pricing_basis_usd_per_b200_hour": 6.25,
}
for k, v in pins.items():
    print(f"{k:20s}: {v}")

## Why no `--moe-backend` flag (root cause, source-supported)

Two earlier attempts used Libertai's per-model image
(`vllm/vllm-openai:glm53-flash-x86_64-cu130`, proven on 4x RTX PRO 6000 / DGX
Spark) plus their documented GLM-5.3 fix-up env vars
(`VLLM_GLM53_MOE_INPUT_SCALE=1.0`, `VLLM_GLM53_CUDA_SPARSE_MLA=1`,
`--moe-backend flashinfer_cutlass`, `NCCL_MIN_NCHANNELS=32`,
`NCCL_P2P_LEVEL=PXB`). Both produced token-0 `"!"` repeated to `max_tokens`
on B200 -- clean boot, clean health check, garbage generation.

Root cause: [vllm-project/vllm#54189](https://github.com/vllm-project/vllm/issues/54189).
This checkpoint is weight-only NVFP4 (ModelOpt) and ships no
`w13_input_scale`; any MoE backend that reads that tensor for
dequantization gets an uninitialized (zero) scale and zeroes every expert's
output. Revision `11d73216` **predates** the checkpoint's
`model-input-scales.safetensors` file (added upstream 2026-08-28/30, after
this pin), so the fix-up env vars -- designed for the newer checkpoint
revision -- do not help, and forcing `--moe-backend flashinfer_cutlass`
reproduces the same zeroed-expert failure.

The fix: drop the per-model image and the env vars, use the stock vLLM
nightly, and let vLLM auto-select the MoE backend. On B200 (sm_100), that
auto-selection lands on Marlin -- the weight-only path, which never reads
`w13_input_scale`. This finding is recorded directly in the operator's
launch script, `modal_regen_glm.py` (`_boot_server` docstring): *"auto-selects
MARLIN, the weight-only path, since B200 lacks native FP4 tensor cores for
the FlashInfer/CUTLASS paths"* -- the operator's own contemporaneous reading
of the server log (**label: measured-by-operator**; the raw server log line
was not preserved for this receipt, only the script's note recorded at the
time).

In [ ]:
# --- Sanity probe replay (measured) ---
# Three short prompts sent at production sampling params before any bulk run.
sanity_probe_prompts = [
    ("math", "What is 17 * 24? Show your work briefly, then give the final number."),
    ("code", "Write a Python function that returns the nth Fibonacci number using iteration, not recursion."),
    ("chat", "In two or three sentences, recommend a good first book for someone new to science fiction."),
]

# Result recorded by the operator on the stock-image boot (job 3, 2026-09-06):
sanity_probe_result = {
    "math": {"ok": True, "note": "correct, coherent, clean </think> close"},
    "chat": {"ok": True, "note": "correct, coherent, clean </think> close"},
    "code": {
        "ok": True,
        "note": ("flagged only because max_tokens cut off mid-reasoning before "
                 "</think>; the reasoning itself was real, on-topic, correct "
                 "Fibonacci discussion (distinct_chars=56) -- NOT the degenerate "
                 "single-repeated-character failure the per-model image produced."),
    },
}
for label, prompt in sanity_probe_prompts:
    print(f"[{label}] {prompt}")
    print(f"  -> {sanity_probe_result[label]}")

## Results (measured + inferred)

- **Boot: 275 s**, weights already local on the volume. Compare to the two
  failed per-model-image attempts: 1,526 s (job 1) and 841 s (job 2) -- the
  stock image skips the per-model image's build/kernel overhead entirely.
- **Bulk throughput ~2,100-2,400 output tok/s at c=128, thinking on**
  (**inferred**): the generation client
  (`DeepSpec/scripts/data/generate_train_data.py`, unmodified) prints no
  throughput figure. The number here is a chars/4-over-assistant-content
  estimate logged periodically during the bulk run -- not a token-accounted
  benchmark like the Qwen3.8 calibration sweep in the sibling folder.
- **Mean answer length ~4.7k characters** (thinking on, unfiltered).
- **c=1, c=8, c=16, and MTP-3 are pending** -- no receipt exists for any of
  them.

## Other pitfalls (measured)

- **c=256 + `--kv-cache-dtype fp8` reached ~2,200 tok/s then the server
  stopped answering at ~20 min** (measured; root cause unknown at time of
  writing). The generation client does not detect a dead server on its own
  and continues logging connection-error rows for the rest of the input.
  A health-check watchdog (kill the generation subprocess after 3
  consecutive `/health` failures) was added after this incident -- see
  `_health_watchdog` in [`modal_regen_glm.py`](modal_regen_glm.py).
- **`--max-model-len 8192` is too small.** With this checkpoint's
  8,192-token output budget, 8192 leaves no room for a multi-turn prompt;
  `--max-model-len 20480` (12k of prompt headroom) is the value pinned
  above -- the same lesson the Qwen3.8 lane in this repo hit at the same
  context size, independently.
- **Thinking cannot be disabled.** The chat template emits `<think>`
  unconditionally regardless of `enable_thinking`/`disable_thinking`
  request flags.

## Reproduce -- Modal path

Requires a Modal account and a volume with the pinned revision already
snapshotted (GPU functions in this recipe never download weights -- a
separate CPU-only `prefetch` function does that; see
[`modal_regen_glm.py`](modal_regen_glm.py)).

In [ ]:
# Dry run: boots the server, runs the sanity probe, generates 20 samples
# modal run modal_regen_glm.py --dry-run

# Full bulk run (50k prompts, resumable)
# modal run --detach modal_regen_glm.py \
#   --input-path /vol/prompts_50k.jsonl \
#   --output-path /vol/regen/glm53_nvfp4/perfectblend_50k_regen.jsonl

print("see modal_regen_glm.py in this folder for the full script and post-mortems")

## Reproduce -- bare-metal / non-Modal path

On any 2x B200 host with Docker and the NVIDIA Container Toolkit:

In [ ]:
launch_cmd = """
docker run --gpus '"device=0,1"' --rm -p 127.0.0.1:8000:8000 --ipc=host \\
  vllm/vllm-openai@sha256:41d42cfabd3289f40fdca71b4a0fe290474880d20e9534a90c698eb78e3eecee \\
  --model LibertAIDAI/GLM-5.3-Flash-NVFP4 \\
  --revision 11d73216cd636238e82e1d77fe1042ffab36e7fa \\
  --tensor-parallel-size 2 \\
  --max-model-len 20480 \\
  --max-num-seqs 128 \\
  --gpu-memory-utilization 0.90 \\
  --trust-remote-code
"""
print(launch_cmd)

In [ ]:
# ALWAYS run the sanity probe below before trusting any bulk output on new
# hardware/checkpoint/image combinations -- this is exactly the gate that
# caught the per-model-image failure before it reached a full corpus run.
import json, urllib.request

def one_request(prompt_text, max_tokens=1024):
    body = json.dumps({
        "model": "LibertAIDAI/GLM-5.3-Flash-NVFP4",
        "messages": [{"role": "user", "content": prompt_text}],
        "temperature": 1.0, "top_p": 0.95, "max_tokens": max_tokens,
    }).encode()
    req = urllib.request.Request(
        "http://127.0.0.1:8000/v1/chat/completions", data=body,
        headers={"Content-Type": "application/json"},
    )
    with urllib.request.urlopen(req, timeout=120) as resp:
        return json.loads(resp.read())

def looks_coherent(content):
    from collections import Counter
    if not content:
        return False
    counts = Counter(content)
    _, top_n = counts.most_common(1)[0]
    return (top_n / len(content)) < 0.4 and len(content.split()) > 15

# Example (requires a live server -- not executed by CI/replay):
# for label, prompt in sanity_probe_prompts:
#     out = one_request(prompt)
#     content = out["choices"][0]["message"]["content"]
#     print(label, "coherent:", looks_coherent(content), "has_think_close:", "</think>" in content)
print("point this at a live server before any bulk run")